# Embedding Fundamentals

**Module:** 01 — Embeddings

Embeddings turn unstructured data into geometry. This lesson builds a practitioner-grade mental model from raw text to production embedding pipelines.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define embeddings and explain why semantic geometry beats keyword matching
- Contrast bag-of-words, TF-IDF, dense, and sparse vector representations
- Distinguish token, sentence, chunk, and document embeddings
- Reason about dimensionality, normalization, and the end-to-end embedding pipeline
- Identify failure modes that break retrieval quality in RAG systems


## What are Embeddings?

**Definition.** An **embedding** is a fixed-length numerical vector that places an item (token, sentence, image, audio clip, product, user) into a continuous vector space so that **semantic relatedness ≈ geometric proximity**.

**Why it matters.** Computers cannot natively compare meaning. Embeddings are the bridge from language/media to math used by search, RAG, clustering, recommendations, and multimodal retrieval.

**How it works.** A neural encoder (or classical factorization model) maps input → ℝᵈ. Training pulls related items together and pushes unrelated items apart (contrastive, MLM, or ranking objectives).

**Intuition.** Think of a city map where neighborhoods are topics. 'King' and 'queen' live on the same block; 'apple' (fruit) is across town. Distance on the map is a proxy for meaning—not spelling.

**Common pitfalls.**
- Assuming cosine similarity always equals 'truth' — it reflects training data biases
- Comparing vectors from different models or incompatible spaces
- Forgetting to normalize when using cosine/dot-product retrieval
- Embedding huge documents as one vector and losing local detail

**When to use.** Use embeddings whenever you need semantic search, near-duplicate detection, clustering, classification with few labels, or RAG retrieval.

### Key terms

- **vector space**: A d-dimensional numeric space where each axis is a learned feature
- **encoder**: Model that maps inputs to embeddings
- **similarity**: Score saying how close two vectors are (cosine, dot, etc.)

```mermaid
flowchart LR
  A[Raw text / image / audio] --> B[Tokenizer / preprocessor]
  B --> C[Encoder model]
  C --> D[Dense vector in R^d]
  D --> E[Index / compare / cluster]
```


In [ ]:
# Demo 1 — meaning as geometry (hand-crafted 2D space)
import numpy as np
import matplotlib.pyplot as plt

words = {
    "king": np.array([0.80, 0.90]),
    "queen": np.array([0.75, 0.85]),
    "man": np.array([0.70, 0.20]),
    "woman": np.array([0.65, 0.15]),
    "apple": np.array([-0.80, -0.10]),
    "orange": np.array([-0.75, -0.20]),
    "car": np.array([-0.10, 0.70]),
    "truck": np.array([-0.05, 0.65]),
}

fig, ax = plt.subplots(figsize=(7, 5.5))
for w, v in words.items():
    ax.scatter(v[0], v[1], s=90)
    ax.annotate(w, (v[0] + 0.02, v[1] + 0.02))
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_title("Toy semantic space — clusters emerge by meaning")
ax.set_xlabel("dim 0")
ax.set_ylabel("dim 1")
plt.tight_layout()
plt.show()


In [ ]:
# Demo 2 — nearest neighbors in the toy space
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

query = "king"
qv = words[query]
ranked = sorted(((cosine(qv, v), w) for w, v in words.items() if w != query), reverse=True)
print(f"Nearest to '{query}':")
for score, w in ranked[:4]:
    print(f"  {w:8s}  cosine={score:.3f}")


### Try it yourself — Semantic neighborhoods

1. Add three domain terms from your work (e.g., 'invoice', 'refund', 'SKU') as 2D points.
2. Place synonyms near each other and unrelated terms far apart.
3. Query one term and verify the nearest neighbors match your intuition.


## Why Needed?

**Definition.** Keyword systems match surface forms. Embedding systems match **intent and paraphrase**, which is what users actually type.

**Why it matters.** Synonyms, typos, multilingual queries, and long natural-language questions defeat exact-term matching. Modern GenAI apps (especially RAG) depend on semantic retrieval quality.

**How it works.** Replace or augment lexical retrieval with vector similarity over embeddings. Often hybridize: BM25 for exact terms + dense vectors for semantics.

**Intuition.** 'Car' and 'automobile' share almost no characters but should retrieve the same policy paragraph. Embeddings collapse that surface gap.

**Common pitfalls.**
- Replacing lexical search entirely when SKUs, IDs, or legal citations need exact match
- Using embeddings alone for high-precision entity lookup
- Ignoring domain shift — general embeddings underperform on jargon-heavy corpora

**When to use.** Default to embeddings (often hybrid) for FAQ search, knowledge bases, support deflection, and document QA.


In [ ]:
# Demo — bag-of-words fails on paraphrase; embeddings would succeed
import numpy as np

docs = ["the quick brown fox", "a fast auburn canine"]
vocab = sorted({t for d in docs for t in d.split()})
print("Vocab:", vocab)

def bow(text, vocab):
    toks = text.split()
    return np.array([toks.count(w) for w in vocab], dtype=float)

v1, v2 = bow(docs[0], vocab), bow(docs[1], vocab)
cos = float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9))
print("BoW cosine (synonyms, zero overlap):", round(cos, 4), "← unrelated under keywords")
print("A semantic embedder should score these much higher despite no shared tokens.")


In [ ]:
# Demo — TF-IDF still cannot bridge synonyms
from collections import Counter
import math

corpus = [
    "customer asked about refund policy",
    "user inquired regarding money back rules",
    "shipping times for express delivery",
]

def tokenize(s):
    return s.lower().split()

df = Counter()
for doc in corpus:
    for t in set(tokenize(doc)):
        df[t] += 1

def tfidf(doc):
    tf = Counter(tokenize(doc))
    n = len(corpus)
    return {t: (c / sum(tf.values())) * math.log((n + 1) / (df[t] + 1)) + 1 for t, c in tf.items()}

def cosine_sparse(a, b):
    keys = set(a) | set(b)
    num = sum(a.get(k, 0) * b.get(k, 0) for k in keys)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return num / (na * nb + 1e-9)

a, b, c = map(tfidf, corpus)
print("refund vs money-back paraphrase:", round(cosine_sparse(a, b), 4))
print("refund vs shipping (should be lower):", round(cosine_sparse(a, c), 4))
print("Note: paraphrase score stays low without shared rare terms — embeddings fix this.")


## Text Representation

**Definition.** Text representation is how we convert tokens into features for machines: from one-hot and bag-of-words, through TF-IDF, to dense neural embeddings.

**Why it matters.** Your representation choice determines whether synonyms match, how large indexes become, and whether downstream models generalize.

**How it works.** Classical pipelines count terms; neural pipelines contextualize tokens with transformers and pool them into sentence/document vectors.

**Intuition.** One-hot is a giant sparse switchboard. Embeddings are a compressed 'meaning fingerprint' shared across vocabulary.

**Common pitfalls.**
- Keeping BoW for semantic search 'because it's simple' when paraphrase is common
- Mixing representations inconsistently across query and document paths

**When to use.** Use lexical features for exact match & explainability; dense embeddings for semantics; hybrid for production search.

| Representation | Semantics | Sparsity | Typical use |
|---|---|---|---|
| One-hot / BoW | None | Very high | Baselines, classical NLP |
| TF-IDF | Weak (rarity weighting) | High | Lexical search, hybrid |
| Word2Vec/GloVe | Word-level | Dense | Legacy NLP, analysis |
| Transformer embeddings | Contextual / strong | Dense | Modern search & RAG |


In [ ]:
# Build a tiny TF-IDF matrix for intuition
from collections import Counter
import math

docs = [
    "vector databases store embeddings",
    "embeddings power semantic search",
    "relational databases store rows",
]
tokenized = [d.split() for d in docs]
vocab = sorted({t for d in tokenized for t in d})
N = len(docs)
df = {t: sum(t in d for d in tokenized) for t in vocab}

rows = []
for d in tokenized:
    tf = Counter(d)
    row = []
    for t in vocab:
        if tf[t] == 0:
            row.append(0.0)
        else:
            row.append((tf[t] / len(d)) * math.log((N + 1) / (df[t] + 1)) + 1)
    rows.append(row)

print("vocab:", vocab)
for i, row in enumerate(rows):
    print(f"doc{i}", [round(x, 3) for x in row])


## Vector Representation

**Definition.** A vector representation stores each item as an ordered list of floats `[x0, x1, ..., x_{d-1}]` where each dimension encodes a latent feature.

**Why it matters.** Vectors enable efficient linear algebra: similarity, clustering, ANN indexes, and neural network layers all speak 'vectors'.

**How it works.** After encoding, you store vectors in memory/DB, optionally normalize them, and query with distance metrics. Metadata travels alongside for filtering.

**Intuition.** Each dimension is a soft knob (topic, tone, entity type…). You rarely interpret single axes; geometry of the whole vector matters.

**Common pitfalls.**
- Treating unnormalized Euclidean and cosine spaces as interchangeable
- Silently truncating or padding vectors to the wrong dimension

**When to use.** Always, once you choose embeddings — but decide early on metric, normalization, and dimension.


In [ ]:
# Vector ops you will use constantly
import numpy as np

a = np.array([0.2, 0.8, -0.1], dtype=float)
b = np.array([0.25, 0.7, -0.05], dtype=float)

def l2_normalize(v):
    return v / (np.linalg.norm(v) + 1e-9)

print("dot:", round(float(np.dot(a, b)), 4))
print("cosine:", round(float(np.dot(l2_normalize(a), l2_normalize(b))), 4))
print("euclidean:", round(float(np.linalg.norm(a - b)), 4))
print("normalized vectors make dot ≈ cosine — important for ANN indexes")


## Dense vs Sparse Embeddings

**Definition.** **Dense** embeddings are short continuous vectors (e.g., 384–3072 dims, mostly nonzero). **Sparse** embeddings / lexical vectors are high-dimensional with few nonzeros (TF-IDF, SPLADE-style learned sparse).

**Why it matters.** Dense captures paraphrase; sparse captures exact terms and rare identifiers. Production systems often need both.

**How it works.** Dense: transformer encoder + pooling. Sparse: term weights or learned sparse activations. Hybrid retrieval fuses both score lists.

**Intuition.** Dense is a blurry but smart sketch of meaning. Sparse is a precise inventory of words present.

**Common pitfalls.**
- Expecting dense vectors to perfectly match invoice IDs or error codes
- Ignoring sparse signals in legal/medical corpora with critical terminology

**When to use.** Dense for semantic FAQ/RAG; sparse/hybrid when exact tokens matter.

| Property | Dense | Sparse |
|---|---|---|
| Dimensionality | Hundreds–thousands | Tens/hundreds of thousands |
| Synonym handling | Strong | Weak unless expanded |
| Exact term match | Weak | Strong |
| Index type | HNSW / IVF | Inverted index |
| Typical role | Semantic recall | Lexical precision |


In [ ]:
# Contrast sparse bag-of-words vs dense random projection sketch
import numpy as np

vocab = ["refund", "policy", "shipping", "invoice", "agent"]
doc = "refund policy for invoice"
sparse = np.array([1.0 if t in doc.split() else 0.0 for t in vocab])

rng = np.random.default_rng(0)
# Simulate a dense encoder with a fixed random projection of hashed features
feats = np.zeros(16)
for t in doc.split():
    feats[hash(t) % 16] += 1.0
dense = feats / (np.linalg.norm(feats) + 1e-9)

print("sparse:", sparse)
print("dense (16-d sketch):", np.round(dense, 3))
print("nonzero sparse:", int(sparse.sum()), "nonzero dense:", int(np.count_nonzero(dense)))


## Token / Sentence / Document Embeddings

**Definition.** **Token embeddings** represent subwords in context; **sentence embeddings** represent a short unit of meaning; **document embeddings** represent longer content (often via chunking + pooling or hierarchical encoders).

**Why it matters.** Using the wrong granularity loses signal: one vector for a 50-page PDF smears topics; token vectors alone are awkward for search.

**How it works.** RAG typically embeds **chunks** (paragraph-sized). Sentence embeddings help FAQ and short texts. Token embeddings power generative models internally.

**Intuition.** Zoom level on a map: tokens are street corners, sentences are blocks, documents are districts. Retrieval needs the right zoom.

**Common pitfalls.**
- Embedding entire books as a single vector
- Chunks so small they lose pronouns/antecedents
- Mixing query-encoder and document-encoder outputs from asymmetric models incorrectly

**When to use.** Default RAG: chunk embeddings. Short FAQ: sentence embeddings. Model internals/analysis: token embeddings.


In [ ]:
# Granularity demo — mean-pool token vectors vs one doc vector
import numpy as np

rng = np.random.default_rng(42)
# Fake contextual token vectors for two sentences in one document
sent1 = rng.normal(size=(5, 8))  # 5 tokens
sent2 = rng.normal(size=(6, 8))

sent1_emb = sent1.mean(axis=0)
sent2_emb = sent2.mean(axis=0)
doc_emb = np.vstack([sent1, sent2]).mean(axis=0)

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

query = sent1_emb  # query about topic in sentence 1
print("query vs sent1:", round(cos(query, sent1_emb), 3))
print("query vs sent2:", round(cos(query, sent2_emb), 3))
print("query vs whole doc:", round(cos(query, doc_emb), 3), "← diluted by other content")


## Embedding Dimensions

**Definition.** Dimensionality **d** is the length of the embedding vector. Common sizes: 384, 768, 1024, 1536, 3072.

**Why it matters.** Higher d can increase quality up to a point but raises storage, RAM, and ANN latency/cost.

**How it works.** Choose model dimension from the model card. Some APIs let you shorten dimensions (Matryoshka / native truncation) with small quality loss.

**Intuition.** Too few dimensions: crowded map, collisions. Too many: expensive map with diminishing returns.

**Common pitfalls.**
- Comparing 768-d and 1536-d vectors directly
- Forgetting storage math: 1M vectors × 1536 × 4 bytes ≈ 6.1 GB (float32)

**When to use.** Start with a strong mid-size model (384–1024) for prototypes; scale dimension only if evaluation proves need.

```text
+---------------------------------+
| Storage rule of thumb           |
+---------------------------------+
| bytes ≈ N * d * bytes_per_dim   |
| float32 => 4 bytes/dim          |
| float16 => 2 bytes/dim          |
| Add index overhead (HNSW graph) |
+---------------------------------+
```


In [ ]:
# Dimension / storage calculator
def embedding_storage_gb(n_vectors, dim, bytes_per=4, index_overhead=1.5):
    raw = n_vectors * dim * bytes_per
    return (raw * index_overhead) / (1024 ** 3)

for d in [384, 768, 1536, 3072]:
    gb = embedding_storage_gb(1_000_000, d)
    print(f"d={d:4d}  ~{gb:.2f} GB for 1M vectors (float32, 1.5x index overhead)")


## Embedding Pipeline

**Definition.** The embedding pipeline is the full path: ingest → clean → chunk → embed → index → query embed → search → (rerank) → use in application.

**Why it matters.** Most production bugs are pipeline bugs: mismatched models, missing normalization, bad chunking, stale indexes—not the math itself.

**How it works.** Offline batch embed documents; online embed queries with the **same** model and preprocessing. Version model IDs with the index.

**Intuition.** Like a factory: inconsistent ingredients (different encoders) produce incomparable products.

**Common pitfalls.**
- Re-embedding docs with model A while queries use model B
- Not re-indexing after model upgrade
- Skipping text cleanup (HTML, boilerplate) so vectors encode noise

**When to use.** Every embedding-backed feature — treat the pipeline as a product.

```mermaid
flowchart TB
  subgraph offline [Offline indexing]
    I[Ingest docs] --> C[Clean + chunk]
    C --> E[Embed chunks]
    E --> X[Write vector index + metadata]
  end
  subgraph online [Online query]
    Q[User query] --> QE[Embed query]
    QE --> S[ANN / hybrid search]
    S --> R[Optional rerank]
    R --> A[App / LLM context]
  end
  X --> S
```


In [ ]:
# Minimal end-to-end pipeline sketch (hash encoder stands in for a real model)
from dataclasses import dataclass
import numpy as np

@dataclass
class Chunk:
    doc_id: str
    text: str
    vector: np.ndarray

def fake_embed(text: str, dim: int = 32) -> np.ndarray:
    rng = np.random.default_rng(abs(hash(text.lower())) % (2**32))
    v = rng.normal(size=dim)
    return v / (np.linalg.norm(v) + 1e-9)

docs = {
    "d1": "Acme refunds are available within 14 days of purchase.",
    "d2": "Express shipping arrives in 2 business days.",
    "d3": "Reset your password from the account settings page.",
}

# Offline index
chunks = []
for doc_id, text in docs.items():
    for i, sentence in enumerate(text.split(". ")):
        sentence = sentence.strip(". ")
        if sentence:
            chunks.append(Chunk(doc_id, sentence, fake_embed(sentence)))

# Online search
query = "How long do I have to return a product?"
qv = fake_embed(query)
ranked = sorted(((float(np.dot(qv, c.vector)), c) for c in chunks), reverse=True)
print("Query:", query)
for score, c in ranked:
    print(f"  {score:.3f}  [{c.doc_id}] {c.text}")


In [ ]:
# API shape — OpenAI-style embeddings request/response (placeholder key)
import os
import json

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY")
request_body = {
    "model": "text-embedding-3-small",
    "input": ["How long is the refund window?", "shipping SLA"],
}
example_response = {
    "object": "list",
    "data": [
        {"object": "embedding", "index": 0, "embedding": [0.012, -0.034, 0.056]},
        {"object": "embedding", "index": 1, "embedding": [0.001, 0.022, -0.018]},
    ],
    "model": "text-embedding-3-small",
    "usage": {"prompt_tokens": 12, "total_tokens": 12},
}
print("REQUEST:")
print(json.dumps(request_body, indent=2))
print("RESPONSE (truncated vectors):")
print(json.dumps(example_response, indent=2))


## Glossary

- **Embedding / vector**: numeric meaning fingerprint
- **Encoder**: model producing embeddings
- **Chunk**: retrieval unit cut from a document
- **ANN**: Approximate Nearest Neighbor search
- **Normalization**: rescaling vectors to unit length for cosine/dot retrieval
- **Index**: data structure for fast similarity search


## Deep Dive — Operational Checklist

**Definition.** Production embedding systems are contracts: model ID, dimension, metric, prefixes, and preprocessing must match between index and query paths.

**Why it matters.** Silent contract drift is the most common cause of sudden recall collapse.

**How it works.** Store model metadata with the collection; fail closed on mismatch; re-embed on upgrades with a dual-read window if needed.

**Intuition.** Two maps with different projections cannot share GPS coordinates.

**Common pitfalls.**
- Swapping models without reindexing
- Comparing cosine thresholds across models
- Mixing instruction prefixes inconsistently

**When to use.** Every deployment—not just the first prototype.


In [ ]:
# Contract validator
contract = {
    "model": "text-embedding-3-small",
    "dim": 1536,
    "metric": "cosine",
    "normalize": True,
    "query_prefix": "",
    "doc_prefix": "",
}

def validate_vector(vec, contract):
    assert len(vec) == contract["dim"], (len(vec), contract["dim"])
    return True

validate_vector([0.0] * contract["dim"], contract)
print("contract ok", contract["model"])


### Try it yourself — Contract lab

1. Write the contract dict for your preferred open embedding model.
2. Intentionally break the dimension and show the assertion firing.
3. Document who owns re-embedding after a model upgrade.


In [ ]:
# Threshold calibration sketch
pairs = [("rel", 0.84), ("rel", 0.79), ("irr", 0.55), ("irr", 0.71)]
for thr in [0.70, 0.75, 0.80]:
    tp = sum(1 for y,s in pairs if y=="rel" and s>=thr)
    fp = sum(1 for y,s in pairs if y=="irr" and s>=thr)
    print(thr, "tp", tp, "fp", fp)


## Comparison table — practical choices

| Concern | Prefer | Avoid |
|---|---|---|
| Paraphrase FAQ | Dense / hybrid | Keywords alone |
| Invoice IDs | Lexical / hybrid | Dense-only |
| Privacy VPC | Local open model | Unapproved SaaS |
| Fast prototype | Small API/local MiniLM | Giant untested models |


In [ ]:
# Mini bake-off harness
rankings = {
    "modelA": ["d2", "d1", "d3"],
    "modelB": ["d1", "d2", "d3"],
}
gold = {"d1"}
for name, ranked in rankings.items():
    hit = ranked[0] in gold
    print(name, "top1_hit", hit)


```mermaid
flowchart LR
  A[Corpus] --> B[Embed+index]
  C[Query] --> D[Embed]
  D --> E[Search]
  B --> E
  E --> F[Evaluate recall]
  F -->|bad| G[Fix chunking/model/metric]
  F -->|good| H[Ship with monitors]
```


### Try it yourself — End-to-end

1. Build a 10-document toy index with hashed embeddings.
2. Create 5 paraphrase queries and compute Recall@3.
3. Write one monitoring alert you would page on in production.


In [ ]:
import numpy as np

def emb(text, dim=32):
    rng = np.random.default_rng(abs(hash(text.lower())) % (2**32))
    v = rng.normal(size=dim)
    return v / (np.linalg.norm(v) + 1e-9)

docs = {
    "d1": "Refunds are available for 14 days",
    "d2": "Reset your password via email",
    "d3": "Express shipping takes two days",
}
X = {i: emb(t) for i,t in docs.items()}
q = emb("how long can I return an item?")
ranked = sorted(((float(np.dot(q,v)), i) for i,v in X.items()), reverse=True)
print(ranked)
print("recall@1", float(ranked[0][1] == "d1"))


## Summary & Key Takeaways

- Embeddings map meaning into vectors so similar concepts are nearby.
- Lexical features fail on paraphrase; dense vectors (often hybridized) fix that.
- Choose granularity deliberately: chunk embeddings dominate RAG.
- Dimension and metric choices drive cost and recall/precision tradeoffs.
- Pipeline consistency (same model, cleanup, versioning) matters as much as model choice.

### Practice

Embed 20 real FAQ pairs from your domain with a free local model and measure whether paraphrases retrieve the gold answer at k=5.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
